# Forecast baseline y punto de reorden

Del histórico limpio a la tabla de decisión por SKU:

1. Demanda diaria higienizada (`prepare_daily_demand`)
2. Clasificación ABC del último trimestre
3. Features rolling y backtest de la media móvil 30 días
4. Punto de reorden, stock de seguridad y pedido sugerido

El forecast de esta notebook es la media móvil 30 días para todo el catálogo. En el top clase A, `predict` la sustituye por Holt-Winters (ETS); la comparación está en la notebook 04.

Por qué se rellenan ceros y se recortan picos: notebook 03.

> Requiere el paquete instalado en modo editable: `pip install -e .` desde la raíz del repo.


In [1]:
import pandas as pd

from inventario_ecommerce import config
from inventario_ecommerce.dataset import load_transactions, save_processed
from inventario_ecommerce.features import (
    build_rolling_features,
    clean_transactions,
    compute_abc_classification,
    prepare_daily_demand,
    sales_by_product_last_quarter,
)
from inventario_ecommerce.modeling.predict import build_reorder_policy, forecast_30d_baseline
from inventario_ecommerce.modeling.train import temporal_backtest_baseline

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Datos y demanda diaria

La serie de cada SKU incluye los días sin venta (cantidad = 0) desde su primera transacción. Sin eso, la media solo vería días con ticket y sesgaría la demanda al alza.


In [2]:
raw = load_transactions()
clean = clean_transactions(raw)

# El ABC va primero: el cap de picos se calcula dentro de cada clase.
abc = compute_abc_classification(sales_by_product_last_quarter(clean))
daily = prepare_daily_demand(clean, abc=abc)
rolling = build_rolling_features(daily)

latest_features = (
    rolling.sort_values("Date")
    .groupby([config.COL_STOCK_CODE, config.COL_DESCRIPTION], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

print(f"Filas raw:     {len(raw):,}")
print(f"Filas limpias: {len(clean):,}")
print(f"Días-SKU:      {len(daily):,}")
print(f"SKUs:          {latest_features[config.COL_STOCK_CODE].nunique():,}")


Filas raw:     1,067,371
Filas limpias: 1,038,067
Días-SKU:      3,322,199
SKUs:          4,906


## 2. Clasificación ABC

- **A**: hasta el 80% de las ventas acumuladas
- **B**: del 80% al 95%
- **C**: el resto

Además de fijar el nivel de servicio, la clase decide el cap de picos de la celda anterior (notebook 03).


In [3]:
abc["ABCClass"].value_counts()


ABCClass
C    1906
B     803
A     684
Name: count, dtype: int64

## 3. Backtest temporal

Holdout: últimos 30 días. Predicción: media diaria de los 30 días anteriores al corte. El MAE se calcula sobre el calendario completo, así que incluye los días a cero.


In [4]:
sku_metrics, global_metrics = temporal_backtest_baseline(
    daily, horizon_days=30, lookback_days=30
)
global_metrics


,CutoffDate,EvalStartDate,EvalEndDate,GlobalMAE,GlobalMAPE,HorizonDays,LookbackDays
0,2011-11-09,2011-11-10,2011-12-09,4.13,144.13,30,30


## 4. Política de reorden

Supuestos del baseline (el dataset no trae stock ni lead time real):

| Parámetro | Valor |
|-----------|------:|
| Lead time | 14 días |
| Ciclo de revisión | 7 días |
| z (A / B / C) | 3.0 / 3.0 / 3.0 |

`ROP = forecast_daily × LT + z × σ_30d × √LT`

El z = 3.0 lo fija el análisis de coste (notebook 06): bajo margen 40% y posesión 25%/año es el mínimo de coste total, y las tres clases eligen el mismo valor.

Sin `data/raw/stock_on_hand.csv`, `recommended_order_qty` es la demanda del ciclo de revisión (`forecast_daily × 7`). Con la fuente, pasa a `max(0, target − on_hand − on_order)`.


In [5]:
forecast = forecast_30d_baseline(daily, lookback_days=30, horizon_days=30)
policy = build_reorder_policy(latest_features, forecast, abc)

print(f"SKUs con recomendación: {len(policy):,}")
policy.head(20)


SKUs con recomendación: 2,963


,StockCode,Description,ABCClass,TotalSales,forecast_daily,forecast_30d,forecast_model,demand_mean_30d,demand_std_30d,demand_cv_30d,lead_time_demand,safety_stock,reorder_point,target_stock,on_hand,on_order,inventory_position,recommended_order_qty,order_basis,recommendation
0,23084,RABBIT NIGHT LIGHT,A,"56,894.39",239.70,"7,191.00",ma30,239.70,117.67,0.49,"3,355.80",827.71,"4,183.51","5,861.41",NaN,NaN,NaN,"1,677.90",ciclo_revision,Monitoreo diario; evitar quiebres
1,22197,POPCORN HOLDER,A,"27,002.40",227.73,"6,832.00",ma30,227.73,117.70,0.52,"3,188.27",827.92,"4,016.19","5,610.32",NaN,NaN,NaN,"1,594.13",ciclo_revision,Monitoreo diario; evitar quiebres
2,22086,PAPER CHAIN KIT 50'S CHRISTMAS,A,"50,907.49",207.47,"6,224.00",ma30,207.47,112.57,0.54,"2,904.53",791.85,"3,696.38","5,148.65",NaN,NaN,NaN,"1,452.27",ciclo_revision,Monitoreo diario; evitar quiebres
3,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,A,"4,504.03",146.90,"4,407.00",ma30,146.90,107.06,0.73,"2,056.60",753.07,"2,809.67","3,837.97",NaN,NaN,NaN,"1,028.30",ciclo_revision,Monitoreo diario; evitar quiebres
4,22578,WOODEN STAR CHRISTMAS SCANDINAVIAN,A,"3,596.04",145.70,"4,371.00",ma30,145.70,100.93,0.69,"2,039.80",709.99,"2,749.79","3,769.69",NaN,NaN,NaN,"1,019.90",ciclo_revision,Monitoreo diario; evitar quiebres
5,22577,WOODEN HEART CHRISTMAS SCANDINAVIAN,A,"3,541.09",142.63,"4,279.00",ma30,142.63,106.90,0.75,"1,996.87",751.96,"2,748.83","3,747.26",NaN,NaN,NaN,998.43,ciclo_revision,Monitoreo diario; evitar quiebres
6,84879,ASSORTED COLOUR BIRD ORNAMENT,A,"19,298.80",135.40,"4,062.00",ma30,135.40,109.17,0.81,"1,895.60",767.93,"2,663.53","3,611.33",NaN,NaN,NaN,947.80,ciclo_revision,Monitoreo diario; evitar quiebres
7,22952,60 CAKE CASES VINTAGE CHRISTMAS,A,"7,084.41",124.80,"3,744.00",ma30,124.80,100.46,0.80,"1,747.20",706.65,"2,453.85","3,327.45",NaN,NaN,NaN,873.60,ciclo_revision,Monitoreo diario; evitar quiebres
8,85099B,JUMBO BAG RED RETROSPOT,A,"31,101.76",122.60,"3,678.00",ma30,122.60,107.68,0.88,"1,716.40",757.43,"2,473.83","3,332.03",NaN,NaN,NaN,858.20,ciclo_revision,Monitoreo diario; evitar quiebres
9,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,A,"24,317.65",121.27,"3,638.00",ma30,121.27,88.25,0.73,"1,697.73",620.78,"2,318.51","3,167.38",NaN,NaN,NaN,848.87,ciclo_revision,Monitoreo diario; evitar quiebres


## 5. Artefactos

La tabla de reorden **canónica** la escribe el CLI, que además aplica ETS en clase A:

```bash
python -m inventario_ecommerce.modeling.predict
```

Aquí se guardan solo los intermedios, para no sobrescribir esa salida con la versión de media móvil.


In [6]:
save_processed(abc, "abc_last_quarter.csv")
save_processed(latest_features, "sku_rolling_features_latest.csv")
save_processed(sku_metrics, "forecast_backtest_by_sku.csv")
save_processed(global_metrics, "forecast_backtest_global.csv")

print("Carpeta:", config.PROCESSED_DATA_DIR)


Carpeta: C:\code\proyecto-a-inventario-ecommerce\data\processed


Salida del CLI: `data/processed/inventory_reorder_recommendations.csv`.

Campos clave: `ABCClass`, `forecast_30d`, `forecast_model`, `safety_stock`, `reorder_point`, `target_stock`, `recommended_order_qty`.
